# Import 

In [ ]:
# COMPUTER VISION TASK:
# Dựa vào bộ dữ liệu CIFAR 10: https://www.cs.toronto.edu/~kriz/cifar.html
# xây dựng 1 mô hình classification tương ứng (dùng sklearn hoặc thư viện ML khác như XGBoost)
# 1. Đọc dữ liệu, 2. Biến đổi dữ liệu thành vector (word2vec, tfidf, làm phẳng ảnh, ….)
# 3. Giảm chiều dữ liệu với PCA hoặc t-SNE, 4. Trực quan hóa

# Import libraries:
import numpy as np
from tensorflow.keras.datasets import cifar10
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Import CIFAR dataset:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

# Flatten data:
x_train_flattened = x_train.reshape(len(x_train), -1)
x_test_flattened = x_test.reshape(len(x_test), -1)

# Normalize data:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_flattened)
x_test_scaled = scaler.transform(x_test_flattened)

# Reduce dimensions with PCA:
pca = PCA(n_components = 50, random_state = 100)
x_train_reduced = pca.fit_transform(x_train_scaled)
x_test_reduced = pca.transform(x_test_scaled)

# Reduce training size to run faster (use only 30% of the training size):
x_train_small, _, y_train_small, _ = train_test_split(
    x_train_reduced, y_train, test_size = 0.7, random_state = 100)

# Train model with XGBoost:
model = XGBClassifier(
    n_estimators = 30,
    max_depth = 5,
    use_label_encoder = False,
    eval_metric = 'mlogloss',
    verbosity = 1
)
model.fit(x_train_small, y_train_small)

# Predict with test set:
y_pred = model.predict(x_test_reduced)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Test set accuracy: {accuracy:.4f}")

In [ ]:
# Trực quan hóa dữ liệu trong không gian 2/3 chiều (không sử dụng label dữ liệu) theo các bước

# Visualize PCA-reduced data using t-SNE (only for x_train)
tsne = TSNE(n_components = 2, perplexity = 30, random_state = 100)
x_train_tsne = tsne.fit_transform(x_train_reduced[:1000])

plt.figure(figsize = (8, 6))
plt.scatter(x_tsne[:, 0], x_tsne[:, 1], s = 5, alpha = 0.6)
plt.title("CIFAR-10 Visualization")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True)
plt.tight_layout()
plt.show()